# Ensemble Adaptativo AIS — Escenario 3

**Pipeline experimental (sin leakage):**

| Partición | Uso |
|-----------|-----|
| `df_70` (70%) | Entrenamiento de modelos base |
| `df_30` → `ens_fit` (50%) | Entrenamiento del AIS (anticuerpos, memoria, NegSel) |
| `df_30` → `held_out` (50%) | Evaluación final — **nadie** lo toca hasta aquí |

**Escenarios comparados:**
1. **Modelos individuales** — TCN, NBEATS, LSTM, TiDE, EncDec, iTransformer
2. **Ensemble estático** — Promedio, Pesos optimizados, Stacking Ridge
3. **Ensemble AIS** — Red inmune artificial adaptativa por sujeto (aiNet + NegSel + SubjectMemory)

In [ ]:
!pip install -q keras-tcn pandas scikit-learn scipy dtaidistance matplotlib joblib openpyxl pyyaml

In [ ]:
import os
import sys
import importlib
import shutil
import glob

# ---- Colab: clonar repo y localizar nueva_info/ ----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.chdir('/content')
    if not os.path.exists('heartrate-forecasting'):
        !git clone https://github.com/AllanDBB/heartrate-forecasting.git
    os.chdir('heartrate-forecasting')

    repo_nueva = os.path.join(os.getcwd(), 'nueva_info')
    os.makedirs(repo_nueva, exist_ok=True)

    search_dirs = [
        '/content/nueva_info',
        '/content/drive/MyDrive/nueva_info',
        '/content/drive/MyDrive',
    ]
    for sdir in search_dirs:
        if not os.path.isdir(sdir):
            continue
        for f in glob.glob(os.path.join(sdir, '*.keras')):
            dst = os.path.join(repo_nueva, os.path.basename(f))
            if not os.path.exists(dst):
                shutil.copy2(f, dst)
                print(f'  Copiado: {f} -> {dst}')

    found = glob.glob(os.path.join(repo_nueva, '*.keras'))
    if len(found) < 6:
        print(f'\n⚠️  Solo se encontraron {len(found)}/6 modelos .keras')
        from google.colab import files
        uploaded = files.upload()
        for fname, data in uploaded.items():
            dest = os.path.join(repo_nueva, fname)
            with open(dest, 'wb') as fh:
                fh.write(data)
            print(f'  Guardado: {dest} ({len(data)/1e6:.1f} MB)')

    found = glob.glob(os.path.join(repo_nueva, '*.keras'))
    print(f'\nModelos en nueva_info/: {len(found)}/6')
    for f in sorted(found):
        print(f'  {os.path.basename(f):20s} {os.path.getsize(f)/1e6:.1f} MB')
    assert len(found) >= 6, f'Faltan modelos. Encontrados: {[os.path.basename(f) for f in found]}'

if not IN_COLAB:
    try:
        _nb_path = __vsc_ipynb_file__
        REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath(_nb_path)))
    except NameError:
        REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
else:
    REPO_DIR = os.getcwd()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'\nREPO_DIR: {REPO_DIR}')
print(f'Colab: {IN_COLAB}')

import main
import utils
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

importlib.reload(utils)
importlib.reload(main)

## 1. Particiones y preparación de datos

In [ ]:
INPUT_SIZE    = 200
OUTPUT_SIZE   = 200
CACHE_DIR     = 'cache_ais'
ENSEMBLE_SEED = 123   # mismo seed que en ensemble_nueva_info para reproducir el split

main.ensure_dir(CACHE_DIR)

df_70, df_30, split_meta = main.load_split_dataframes(
    dataset_dir='dataset',
    split_seed=42,
    split_70_path='nueva_info/df_70.csv',
    split_30_path='nueva_info/df_30.csv',
)
df_70, df_30, overlap = utils.sanitize_split_dataframes(df_70, df_30)
print('Overlap eliminado:', overlap)

path_est_70 = os.path.join(CACHE_DIR, 'values_deses_70.csv')
df_scaled_70, params_70 = utils.estandarizar(df_70, path_est_70)

print(f'\ndf_70 estandarizado: {df_scaled_70.shape}')

In [ ]:
# Cargar particiones pre-split desde nueva_info/ (split por sujeto, sin leakage)
df_tunning_raw = pd.read_csv(os.path.join('nueva_info', 'df_30_tunning.csv'))
df_eval_raw    = pd.read_csv(os.path.join('nueva_info', 'df_30_eval.csv'))

# Estandarizar cada partición con sus propios parámetros
path_est_tunning = os.path.join(CACHE_DIR, 'values_deses_tunning.csv')
path_est_eval    = os.path.join(CACHE_DIR, 'values_deses_eval.csv')
df_scaled_tunning, params_tunning = utils.estandarizar(df_tunning_raw, path_est_tunning)
df_scaled_eval,    params_eval    = utils.estandarizar(df_eval_raw,    path_est_eval)

# Generar ventanas supervisadas
X_ens_fit, y_ens_fit, ids_ens_fit = utils.series_to_supervised_matrix(
    df_scaled_tunning, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE
)
X_held_out, y_held_out, ids_held_out = utils.series_to_supervised_matrix(
    df_scaled_eval,   input_size=INPUT_SIZE, output_size=OUTPUT_SIZE
)

ids_ens_fit  = np.array(ids_ens_fit)
ids_held_out = np.array(ids_held_out)

print('--- Particiones pre-split (nueva_info/) ---')
print(f'ens_fit  (df_30_tunning.csv): X={X_ens_fit.shape}, y={y_ens_fit.shape}')
print(f'held_out (df_30_eval.csv):    X={X_held_out.shape}, y={y_held_out.shape}')
print(f'Sujetos en ens_fit:  {len(set(ids_ens_fit))}')
print(f'Sujetos en held_out: {len(set(ids_held_out))}')

## 2. Inferencia de modelos base

In [ ]:
import wrappers.KerasPretrainedWrapper as _kpw_mod
import wrappers.NBeatsSupervisedWrapper as _nbeats_mod
importlib.reload(_nbeats_mod)
importlib.reload(_kpw_mod)

# Registro de clases personalizadas en el registry de Keras
_nbeats_mod._get_nbeats_block_class()

from wrappers.KerasPretrainedWrapper import KerasPretrainedWrapper

MODEL_SPECS = {
    'TCN':          'nueva_info/tcn.keras',
    'NBEATS':       'nueva_info/nbeats.keras',
    'LSTM':         'nueva_info/lstm.keras',
    'TiDE':         'nueva_info/tide.keras',
    'EncDec':       'nueva_info/encDec.keras',
    'iTransformer': 'nueva_info/itrans.keras',
}

print('Modelos a cargar:')
for name, path in MODEL_SPECS.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f'  {name:15s} -> {path} ({size_mb:.1f} MB) {"OK" if exists else "MISSING"}')

In [ ]:
# Predicciones sobre AMBAS particiones (en escala estandarizada)
preds_ens_fit_std  = {}   # sobre ens_fit  — para entrenar AIS
preds_held_out_std = {}   # sobre held_out — para evaluar AIS

for name, model_path in MODEL_SPECS.items():
    try:
        wrapper = KerasPretrainedWrapper(model_path, batch_size=32, name=name).load()
        preds_ens_fit_std[name]  = wrapper.predict(X_ens_fit)
        preds_held_out_std[name] = wrapper.predict(X_held_out)
        print(f'[OK] {name}: ens_fit={preds_ens_fit_std[name].shape}, held_out={preds_held_out_std[name].shape}')
    except Exception as exc:
        print(f'[SKIP] {name}: {exc}')

print(f'\nModelos cargados: {len(preds_ens_fit_std)} / {len(MODEL_SPECS)}')

In [ ]:
# Desestandarizar a escala original (bpm)
y_ens_fit_orig  = utils.desestandarizar_ventanas(y_ens_fit,  ids_ens_fit,  params_tunning)
y_held_out_orig = utils.desestandarizar_ventanas(y_held_out, ids_held_out, params_eval)

preds_ens_fit_orig  = {n: utils.desestandarizar_ventanas(p, ids_ens_fit,  params_tunning)
                       for n, p in preds_ens_fit_std.items()}
preds_held_out_orig = {n: utils.desestandarizar_ventanas(p, ids_held_out, params_eval)
                       for n, p in preds_held_out_std.items()}

print('Escala original restaurada.')
print(f'  y_held_out_orig: {y_held_out_orig.shape}  rango [{y_held_out_orig.min():.1f}, {y_held_out_orig.max():.1f}] bpm')

## 3. Métricas individuales (Escenario 1)

In [ ]:
print('=== Escenario 1: Modelos individuales (evaluados sobre held_out) ===')
individual_results = {}
for name, pred in preds_held_out_orig.items():
    individual_results[name] = utils.evaluate_all_metrics(y_held_out_orig, pred)

df_individual = pd.DataFrame(individual_results).T
df_individual.index.name = 'Modelo'
df_individual = df_individual.sort_values('MAPE')
df_individual

## 4. Resultados del ensemble estático (Escenario 2 — referencia)

In [ ]:
# Resultados del ensemble estático (ya calculados en ensemble_nueva_info.ipynb — ventana 200)
# Se incluyen aquí para la tabla comparativa final sin necesidad de re-ejecutar.
ensemble_results = {
    'Ensemble (Promedio)':       {'MAPE': 5.3829, 'MAPE_median': 4.6495, 'DTW': 98.1028,  'Pearson': 0.7675, 'Pearson_median': 0.8912},
    'Ensemble (Pesos Optimos)':  {'MAPE': 4.9767, 'MAPE_median': 4.1849, 'DTW': 84.9812,  'Pearson': 0.7741, 'Pearson_median': 0.8959},
    'Ensemble (Stacking)':       {'MAPE': 4.2210, 'MAPE_median': 3.7068, 'DTW': 66.6905,  'Pearson': 0.8371, 'Pearson_median': 0.9190},
}
print('Resultados ensemble estático cargados desde ensemble_nueva_info.ipynb (ventana=200):')
for k, v in ensemble_results.items():
    print(f"  {k:30s}  MAPE={v['MAPE']:.4f}  Pearson={v['Pearson']:.4f}")

## 5. Ensemble AIS adaptativo (Escenario 3)

El AIS se entrena completamente sobre `ens_fit`:
- **FeatureExtractor**: extrae 20 features estadístico-espectrales por ventana
- **AiNetCore**: aprende K anticuerpos (centroides + pesos de combinación) via selección clonal
- **SubjectMemory**: calibra la memoria inmunológica M_i por sujeto según qué anticuerpo predice mejor para cada persona
- **NegativeSelector**: detecta ventanas fuera de distribución (nonself) via KDTree

En inferencia sobre `held_out`:
- Ventanas **self** → combinación adaptativa ponderada por afinidad × memoria del sujeto
- Ventanas **nonself** → fallback al mejor modelo individual (TCN)

In [ ]:
import importlib
import wrappers.AISEnsembleWrapper as _ais_mod
import wrappers.AiNetCore as _ainet_mod
import wrappers.FeatureExtractor as _fe_mod
import wrappers.SubjectMemory as _sm_mod
import wrappers.NegativeSelector as _ns_mod
for _m in [_fe_mod, _ainet_mod, _sm_mod, _ns_mod, _ais_mod]:
    importlib.reload(_m)

from wrappers.AISEnsembleWrapper import AISEnsembleWrapper

# Hiperparámetros del AIS
AIS_PARAMS = dict(
    n_antibodies=30,          # K anticuerpos en la red
    sigma=1.0,                # ancho del kernel gaussiano de afinidad
    clone_factor=5,           # clones por anticuerpo seleccionado (beta)
    suppression_threshold=0.3,# umbral de supresión de red (sigma_s)
    n_new=5,                  # anticuerpos aleatorios nuevos por iteración
    max_iter=50,              # iteraciones del algoritmo aiNet
    mutation_rate=0.1,        # tasa base de mutación hipersomática
    neg_sel_percentile=99.0,  # percentil para umbral de NegativeSelector
    random_state=42,
)
print('Hiperparámetros AIS:')
for k, v in AIS_PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# Construir y entrenar el AIS
# NOTA: add_model recibe predicciones en escala ORIGINAL (desestandarizada)
# El AIS usa SubjectMemory para evaluar qué anticuerpo predice mejor por sujeto
# con MAPE en escala original. X_ens_fit se usa solo para extraer features.

ais = AISEnsembleWrapper(**AIS_PARAMS)

# Registrar predicciones de modelos base en escala original
for name in preds_ens_fit_orig:
    ais.add_model(name, preds_ens_fit_orig[name], split='fit')
    ais.add_model(name, preds_held_out_orig[name], split='eval')

print('Entrenando AIS sobre ens_fit...')
print(f'  X_ens_fit: {X_ens_fit.shape} (features extraídas internamente)')
print(f'  y_ens_fit_orig: {y_ens_fit_orig.shape}')
print(f'  ids_ens_fit: {ids_ens_fit.shape}  ({len(set(ids_ens_fit))} sujetos únicos)')
print()

ais.fit(
    X_fit=X_ens_fit,           # ventanas estandarizadas (para FeatureExtractor)
    y_true_fit=y_ens_fit_orig, # ground truth en escala original (para MAPE)
    ids_fit=ids_ens_fit,       # IDs de sujeto
    fallback_model='TCN',      # mejor modelo individual para ventanas nonself
)

print('\nAIS entrenado.')
print(f'  Anticuerpos aprendidos: {ais.ainet_.K}')
print(f'  centroids_ shape: {ais.ainet_.centroids_.shape}')
print(f'  weights_    shape: {ais.ainet_.weights_.shape}')
print(f'  Sujetos en memoria: {list(ais.subject_memory_._memory.keys())[:5]} ...')
print(f'  Umbral NegSel: {ais.neg_selector_.threshold:.4f}')

In [ ]:
# Inferencia AIS sobre held_out
print('Prediciendo sobre held_out...')
y_ais_pred = ais.predict(X_eval=X_held_out, ids_eval=ids_held_out)

print(f'y_ais_pred shape: {y_ais_pred.shape}')
print(f'¿Finito? {np.isfinite(y_ais_pred).all()}')
print(f'Rango: [{y_ais_pred.min():.1f}, {y_ais_pred.max():.1f}] bpm')

ais_metrics = utils.evaluate_all_metrics(y_held_out_orig, y_ais_pred)
print(f"\nMAPE AIS: {ais_metrics['MAPE']:.4f}")

## 6. Tabla comparativa final — 3 escenarios

In [ ]:
all_results = {
    **individual_results,
    **ensemble_results,
    'Ensemble (AIS)': ais_metrics,
}

df_final = pd.DataFrame(all_results).T
df_final.index.name = 'Modelo / Método'
df_final = df_final.sort_values('MAPE')

# Resaltar la fila del AIS
print('=== TABLA FINAL — Evaluación sobre held_out (nunca visto) ===')
print(f'{"Modelo / Método":35s}  {"MAPE":>7s}  {"Pearson":>8s}  {"DTW":>9s}')
print('-' * 70)
for idx, row in df_final.iterrows():
    marker = ' <-- AIS' if 'AIS' in str(idx) else ''
    print(f'{str(idx):35s}  {row["MAPE"]:7.4f}  {row["Pearson"]:8.4f}  {row["DTW"]:9.4f}{marker}')

df_final

## 7. Diagnóstico del sistema AIS

Análisis cualitativo del comportamiento inmunológico:
- ¿Qué fracción de ventanas son detectadas como **nonself** (fuera de distribución)?
- ¿Qué sujetos presentan más anomalías?
- ¿Cuál es la distribución de pesos adaptativos por modelo?

In [ ]:
diag = ais.get_diagnostics(X_eval=X_held_out, ids_eval=ids_held_out)

print(f"Fracción de ventanas nonself (fuera de distribución): {diag['nonself_ratio']:.2%}")
print(f"  → {diag['nonself_ratio'] * len(X_held_out):.0f} ventanas de {len(X_held_out)} total\n")

print('Fracción nonself por sujeto:')
ns_by_subj = diag['nonself_by_subject']
for subj, ratio in sorted(ns_by_subj.items(), key=lambda x: -x[1])[:10]:
    bar = '█' * int(ratio * 40)
    print(f'  {subj:30s} {ratio:5.1%}  {bar}')

print('\nPesos adaptativos medios por modelo (ventanas self):')
model_names = diag['model_names']
mean_w = diag['mean_adaptive_weights']
std_w  = diag['weight_std']
for name, w, s in sorted(zip(model_names, mean_w, std_w), key=lambda x: -x[1]):
    bar = '█' * int(w * 40)
    print(f'  {name:15s} {w:.3f} ± {s:.3f}  {bar}')

In [ ]:
# Memoria inmunológica por sujeto
print('Memoria inmunológica M_i por sujeto (pesos sobre anticuerpos):')
print('  Cada fila = distribución de probabilidad sobre K anticuerpos')
print()

unique_subjs = sorted(set(ids_held_out))[:8]  # primeros 8 sujetos
mem_matrix = ais.subject_memory_.get_memory_matrix(np.array(unique_subjs))
df_memory = pd.DataFrame(
    mem_matrix,
    index=unique_subjs,
    columns=[f'Ab_{k}' for k in range(ais.ainet_.K)]
)
df_memory.index.name = 'Sujeto'
df_memory.round(3)

## 8. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# --- Comparación de métricas: todos los métodos ---
utils.plot_metrics_comparison(all_results, title='Comparación — held_out (3 escenarios)')

In [ ]:
# --- Muestras de predicción: AIS vs Real ---
utils.plot_forecast_samples(
    y_held_out_orig, y_ais_pred, n_samples=6,
    title='Ensemble AIS vs Real (held_out)'
)

In [ ]:
# --- Error según horizonte ---
utils.plot_error_over_horizon(
    y_held_out_orig, y_ais_pred,
    title='Ensemble AIS: Error según horizonte (held_out)'
)

In [ ]:
# --- Rendimiento por sujeto ---
utils.plot_subject_performance(
    y_held_out_orig, y_ais_pred, ids_held_out,
    title='Ensemble AIS: Rendimiento por sujeto (held_out)'
)

In [ ]:
# --- Distribución de pesos adaptativos por modelo ---
features_held = ais.feature_extractor_.transform(X_held_out)
is_nonself    = ais.neg_selector_.predict(features_held)
memory_held   = ais.subject_memory_.get_memory_matrix(ids_held_out)
adaptive_w    = ais.ainet_.get_adaptive_weights(features_held, memory_held)

w_self = adaptive_w[~is_nonself]

fig, axes = plt.subplots(1, len(ais._model_names), figsize=(4 * len(ais._model_names), 4), sharey=True)
for ax, name, w_col in zip(axes, ais._model_names, w_self.T):
    rng = w_col.max() - w_col.min()
    if rng < 1e-6:
        ax.bar([w_col.mean()], [len(w_col)], width=0.01, color='steelblue', alpha=0.85)
    else:
        n_bins = max(5, min(30, int(rng / 0.005)))
        ax.hist(w_col, bins=n_bins, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'{name}\nμ={w_col.mean():.3f}')
    ax.set_xlabel('Peso adaptativo')
    ax.set_xlim(0, 1)
axes[0].set_ylabel('Frecuencia')
fig.suptitle('Distribución de pesos adaptativos por modelo (ventanas self)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nVentanas self: {w_self.shape[0]} ({(~is_nonself).mean():.1%})')
print(f'Ventanas nonself (fallback TCN): {is_nonself.sum()} ({is_nonself.mean():.1%})')

## 9. Guardar resultados

In [ ]:
try:
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
except NameError:
    _root = REPO_DIR

_cache_abs = os.path.join(_root, CACHE_DIR)
os.makedirs(_cache_abs, exist_ok=True)

# Tabla final con todos los escenarios
csv_path = os.path.join(_cache_abs, 'final_results_ais.csv')
df_final.to_csv(csv_path)
print(f'Resultados guardados en: {csv_path}')

# Diagnóstico resumido
import json
diag_summary = {
    'nonself_ratio': diag['nonself_ratio'],
    'nonself_by_subject': diag['nonself_by_subject'],
    'mean_adaptive_weights': {
        name: float(w) for name, w in zip(diag['model_names'], diag['mean_adaptive_weights'])
    },
    'weight_std': {
        name: float(s) for name, s in zip(diag['model_names'], diag['weight_std'])
    },
}
diag_path = os.path.join(_cache_abs, 'ais_diagnostics.json')
with open(diag_path, 'w') as f:
    json.dump(diag_summary, f, indent=2)
print(f'Diagnóstico AIS guardado en: {diag_path}')

print('\n=== Resumen final ===')
print(df_final[['MAPE', 'Pearson', 'DTW']].to_string())